# 06: Validating the whole BIDS dataset

**Who this is for:** Users of the Scully Center MEG lab.

This notebook runs the official [BIDS validator](https://bids-standard.github.io/bids-validator/) on your
**whole** BIDS dataset and lists what it finds, **errors first**. Run it once everything is done (Notebooks 01 through 05).

The validator only **reads** the dataset (it is shown the folder read-only), so this notebook changes
nothing. Re-run it whenever you like: after fixing something, or after adding a participant.

**What you will do:**

1. Load the packages and point the notebook at your dataset.
2. Look at what's in the dataset.
3. Run the validator and read its findings.
4. Find which notebook fixes each finding.

> 🖥️ **Where to run this:** on the cluster (scotty). The validator runs from an Apptainer container, and
> containers don't run on a laptop.

### Running this notebook from the pipeline

**Run the two cells below first, also when you run the notebook yourself.** They define `from_config()`,
which the settings cells use. Run by hand, `config_file` stays `None` and they change nothing: the notebook
uses the settings you edit in its cells. They let the Pygers-MEG-to-BIDS pipeline (`run_bids.sh`) run this notebook
with the settings from a study's YAML files instead.

In pipeline mode, each settings cell ends with a line like
`EVENT_ID = from_config("EVENT_ID")`, which replaces the value typed above it with the one from the
YAML files. Run by hand, that line leaves the value as you typed it.

In [ ]:
# Left as None when you run the notebook yourself. The pipeline replaces it with the path of its
# settings file (papermill adds a cell right below this one that sets it).
config_file = None

In [ ]:
# Nothing to edit here. In pipeline mode (config_file set above), this reads this notebook's settings
# from the pipeline's settings file. Run by hand, PIPELINE stays None and from_config() changes nothing.
from pathlib import Path   # file paths that work on macOS, Linux and Windows

NOTEBOOK_ID = "06"     # this notebook's section in the pipeline's settings file
PIPELINE = None            # None = running by hand
if config_file is not None:
    import yaml                                       # reads .yaml files
    with open(config_file, encoding="utf-8") as f:
        PIPELINE = yaml.safe_load(f)[NOTEBOOK_ID]     # a dict: setting name -> value
    print(f"Pipeline mode: notebook {NOTEBOOK_ID} settings from {config_file}")


def from_config(*names):
    """The named settings: from the pipeline's settings file in pipeline mode, unchanged otherwise.

    Example:  EVENT_ID, DROP_UNMAPPED = from_config("EVENT_ID", "DROP_UNMAPPED")

    In pipeline mode every name must be in the settings file: a missing one stops the notebook, so a
    setting can never silently keep this notebook's example value. Settings the file lists under
    "_paths" are turned into Paths.
    """
    if PIPELINE is None:
        values = [globals()[name] for name in names]          # run by hand: keep the values typed above
    else:
        missing = [name for name in names if name not in PIPELINE]
        if missing:
            raise KeyError(f"The pipeline settings file has no value for {missing} (notebook {NOTEBOOK_ID}).")
        paths = set(PIPELINE.get("_paths", []))
        values = [Path(PIPELINE[name]) if name in paths and PIPELINE[name] is not None else PIPELINE[name]
                  for name in names]
    return values[0] if len(values) == 1 else tuple(values)

## 0. Load the Python packages

In [ ]:
# "import x" makes package x usable; "import x as y" lets the code use the short name y;
# "from x import y" brings in only y.
import json                      # read the validator's report, which it prints as JSON
import re                        # text patterns (removes color codes from the validator's text output)
import shutil                    # finds Apptainer, the program that runs the validator container
import subprocess                # runs the validator container, as if typed in a terminal
from pathlib import Path         # Path("...") turns text into a file path that works on macOS, Linux and Windows

import pandas as pd              # tables: the validator's findings are shown as a pandas DataFrame
from IPython.display import display   # shows a table in the output, also in the middle of a cell
from mne_bids import print_dir_tree   # pretty-prints the folders of a BIDS dataset

# Make pandas show every column, and long file names in full.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Settings ✏️ (edit this cell)

Edit the lines between the `EDIT THESE LINES` markers, then run the cell. It checks each path and stops
with a message if one is wrong.

In [ ]:
# ---- EDIT THESE LINES -------------------------------------------------------
# BIDS_ROOT: the top folder of your BIDS dataset (the same BIDS_ROOT as in notebooks 04 and 05).
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")

# VALIDATOR_SIF: the BIDS validator container (pull it once: see the commands at the top of notebook 03).
VALIDATOR_SIF = Path("/jukebox/[yourpath]/singularity-containers/bids-validator/bids-validator-3.0.2.sif")

IGNORE_WARNINGS = False    # True: report errors only
SHOW_TEXT_REPORT = False   # True: also print the validator's own text report (long)
FAIL_ON_ERRORS = False     # True: stop with an error at the end if the validator found errors
SUMMARY_FILE = None        # a .json file to save the error/warning counts in (None = don't save)
# -----------------------------------------------------------------------------

# Pipeline mode: these come from the YAML files (run by hand: unchanged)
BIDS_ROOT, VALIDATOR_SIF, IGNORE_WARNINGS, SHOW_TEXT_REPORT, FAIL_ON_ERRORS, SUMMARY_FILE = from_config(
    "BIDS_ROOT", "VALIDATOR_SIF", "IGNORE_WARNINGS", "SHOW_TEXT_REPORT", "FAIL_ON_ERRORS", "SUMMARY_FILE")

# ---- Check the settings (nothing below here needs editing) ----------------------
# assert stops the notebook with the message if the condition is False.
assert "[yourpath]" not in str(BIDS_ROOT), "Set BIDS_ROOT above (it still contains [yourpath])."
assert BIDS_ROOT.is_dir(), f"BIDS_ROOT not found: {BIDS_ROOT}"
assert (BIDS_ROOT / "dataset_description.json").is_file(), (
    f"No dataset_description.json in {BIDS_ROOT}. Is this the TOP folder of the dataset? "
    "(Notebook 04 writes this file.)")
assert "[yourpath]" not in str(VALIDATOR_SIF), "Set VALIDATOR_SIF above (it still contains [yourpath])."
assert VALIDATOR_SIF.is_file(), (f"Validator container not found: {VALIDATOR_SIF} "
                                 "(see the pull commands at the top of notebook 03).")

# Find Apptainer. shutil.which() returns the full path to a program, or None if it isn't installed.
# Newer clusters call it "apptainer", older ones "singularity"; they work the same.
SING = shutil.which("apptainer") or shutil.which("singularity")
assert SING, ("Apptainer/Singularity not found. Run this notebook on the cluster (e.g. scotty); "
              "containers don't run on a laptop.")

print("Dataset   :", BIDS_ROOT)
print("Validator :", VALIDATOR_SIF)
print("Apptainer :", SING)

## 2. What's in the dataset?

Before validating, a quick look at what the dataset holds: the folder tree (top levels only), and a table
with one row per data folder. Check that every participant and session you expect is there, with an
`anat` folder (from notebooks 03/04) and a `meg` folder with eye-tracking files (from notebook 05).

In [ ]:
# The top levels of the tree: dataset files, then sub-*/ses-*/<datatype> folders.
print_dir_tree(BIDS_ROOT, max_depth=3)

# One row per data folder (anat, meg, ...) of each participant/session, with its number of files.
# rglob("*") lists everything below a folder; we keep the folders named like a BIDS datatype.
DATATYPES = ["anat", "meg", "func", "dwi", "fmap", "beh"]
rows = []
for folder in sorted(BIDS_ROOT.glob("sub-*/**/*")):
    if folder.is_dir() and folder.name in DATATYPES:
        parts = folder.relative_to(BIDS_ROOT).parts       # e.g. ("sub-01", "ses-01", "meg")
        files = [f for f in folder.iterdir() if f.is_file()]
        rows.append({
            "participant": parts[0],
            "session": parts[1] if len(parts) == 3 else "n/a",    # "n/a" if the dataset has no sessions
            "datatype": folder.name,
            "files": len(files),
            # eye-tracking files have "_physio" in their names (they live in meg/, next to the MEG data)
            "eye-tracking files": sum("_physio" in f.name for f in files) if folder.name == "meg" else "",
        })
print()
display(pd.DataFrame(rows))

## 3. Run the BIDS validator

Runs the official validator on `BIDS_ROOT` and lists its findings, **errors first**. Errors make the
dataset invalid and must be fixed (Section 4 says where); warnings are advice.

The validator runs from its container, `VALIDATOR_SIF`, with Apptainer (called Singularity on older
clusters). The dataset is made visible inside the container read-only, so the validator can read it but
never change it.

Warnings you should expect (none of them make the dataset invalid):

> **Split recordings:** a FIF file larger than 2 GB is written as several files (`_split-01_meg.fif`,
> `_split-02_meg.fif`, …). The validator checks each split file, so a per-file warning lists each split
> separately and the total number of warnings goes up. This is expected.

* `TOO_FEW_AUTHORS` (shown for `dataset_description.json`): only one author is listed in `CITATION.cff`
  (notebook 04, 11.4c). It stays if the dataset really has a single author.
* `JSON_KEY_RECOMMENDED` in `dataset_description.json`: `License` (the validator asks for it here even when
  `CITATION.cff` carries it), `HEDVersion` and `SourceDatasets` (not used in this dataset).
* `SIDECAR_KEY_RECOMMENDED` in `_meg.json`, for RECOMMENDED fields that are left out:
  * `DeviceSerialNumber` (notebook 04, 10.1b);
  * `CogAtlasID`, `CogPOID` (notebook 04, 11.4d);
  * on the empty-room itself: `Instructions` and `AssociatedEmptyRoom`;
  * fields that do not apply to OPM or are not recorded: `ContinuousHeadLocalization`, `HeadCoilFrequency`,
    `MaxMovement` (no head coils), `ECOGChannelCount`, `SEEGChannelCount`, `SubjectArtefactDescription`.
* `SIDECAR_KEY_RECOMMENDED` in the `anat/` sidecars (e.g. `SpoilingType`): optional MRI metadata the
  scanner doesn't record (the same warnings notebook 03 showed for its staging folder).

The validator also works outside the notebook, in a terminal on the cluster:

```bash
apptainer run --cleanenv -B <BIDS_ROOT>:/data:ro <VALIDATOR_SIF> /data
```

or in a browser at <https://bids-standard.github.io/bids-validator/> (the folder is read locally, nothing
is uploaded).

In [ ]:
def validator_cmd(*args):
    """The command that runs the validator container, followed by the validator's own arguments.

    run          start the container's own program (the BIDS validator)
    --cleanenv   keep this notebook's environment settings out of the container
    -B ...:ro    show BIDS_ROOT inside the container at /data, read-only ("ro"), so the validator can
                 read the dataset but never change it; the validator is then pointed at /data
    """
    return [SING, "run", "--cleanenv", "-B", f"{BIDS_ROOT.resolve()}:/data:ro", str(VALIDATOR_SIF), *args]


def strip_colors(text):
    """Remove the terminal color codes (such as \x1b[31m for red) from the validator's text."""
    return re.sub(r"\x1b\[[0-9;]*m", "", text)

# Ask the container for the validator's version, as if typed in a terminal, and keep what it prints
# (stdout). This leaves a record in the notebook of which validator checked the dataset.
version = strip_colors(subprocess.run(validator_cmd("--version"), capture_output=True, text=True).stdout).strip()
print(f"{version}  ({VALIDATOR_SIF.name})")

# --- Run it, asking for machine-readable JSON output ---------------------------
cmd = validator_cmd("/data", "--format", "json")   # /data = BIDS_ROOT inside the container
if IGNORE_WARNINGS:
    cmd.append("--ignoreWarnings")
result = subprocess.run(cmd, capture_output=True, text=True)
# Read the printed JSON into a dict. If it is not JSON (e.g. the validator failed), show what it said and stop.
try:
    report = json.loads(result.stdout)
except json.JSONDecodeError:
    print(strip_colors(result.stdout + result.stderr))
    raise RuntimeError(f"The validator did not return a JSON report (exit code {result.returncode}).")

# --- Summary of what the validator found in the dataset --------------------------
s = report["summary"]
print(f"Subjects: {s['subjects']}  Sessions: {s['sessions']}  Tasks: {s['tasks']}  "
      f"Files: {s['totalFiles']}  Schema: {s['schemaVersion']}")

# --- Issues: one row per issue, grouped by severity / code / field ---------------
issues = pd.DataFrame(report["issues"]["issues"])
n_errors = n_warnings = 0
if issues.empty:
    print("\n✓ No errors and no warnings.")
else:
    # Not every issue has a subCode; make missing ones "" so they can be grouped.
    issues["subCode"] = issues.get("subCode", pd.Series(dtype=str)).fillna("")
    # An ordered category, so that errors sort before warnings.
    issues["severity"] = pd.Categorical(issues["severity"], ["error", "warning", "ignore"], ordered=True)
    # One row per (severity, code, subCode): the number of files concerned, and up to 3 of their names.
    table = (issues.groupby(["severity", "code", "subCode"], observed=True)
                   .agg(n_files=("location", "nunique"),
                        files=("location", lambda x: ", ".join(sorted(set(x))[:3]) + (" …" if x.nunique() > 3 else "")))
                   .reset_index())
    n_errors = int((issues["severity"] == "error").sum())
    n_warnings = int((issues["severity"] == "warning").sum())
    print(f"\n{'✗' if n_errors else '✓'} {n_errors} error(s), {n_warnings} warning(s)")
    with pd.option_context("display.max_colwidth", 120):
        display(table)

    # For errors, the full list: every file, with the validator's explanation where it gives one.
    if n_errors:
        cols = [c for c in ["code", "subCode", "location", "issueMessage"] if c in issues.columns]
        print("\nEvery error, one row per file (Section 4 says where to fix each one):")
        with pd.option_context("display.max_colwidth", 200):
            display(issues.loc[issues["severity"] == "error", cols].drop_duplicates().reset_index(drop=True))

if SHOW_TEXT_REPORT:
    text = subprocess.run(validator_cmd("/data"), capture_output=True, text=True)
    print(strip_colors(text.stdout + text.stderr))

# --- Save the counts (the pipeline puts them on its summary page) and stop on errors if asked ---
if SUMMARY_FILE is not None:
    Path(SUMMARY_FILE).write_text(json.dumps({"validator": version, "errors": n_errors, "warnings": n_warnings}) + "\n")
    print(f"\nCounts saved to {SUMMARY_FILE}")
if FAIL_ON_ERRORS and n_errors:
    raise RuntimeError(f"The BIDS validator found {n_errors} error(s): see the table above.")

## 4. Where to fix what

The `location` of each finding is the file it is about. Fix it in the notebook that wrote that file, re-run
that notebook, then re-run this one:

| Files in the finding | Fix it in |
|---|---|
| `dataset_description.json`, `README`, `CHANGES`, `LICENSE`, `CITATION.cff`, `participants.tsv/.json`, `task-<label>_meg.json` | notebook 04, Section 11 |
| `*_meg.json`, `*_channels.tsv/.json`, `*_events.tsv`, `*_coordsystem.json` | notebook 04, Section 10 |
| `*_events.json` | notebook 04 (10.4); notebook 05 merges the eye-tracker's screen fields into it |
| `anat/` files | notebook 03 (the conversion), then notebook 04, Section 12 (the copy) |
| `*_scans.tsv` | notebook 04: the MEG rows (Sections 8–9) and the MRI rows (Section 12) |
| `beh/` files (`*_beh.tsv`, `*_beh.json`) | notebook 04, Section 13 (the PsychoPy logs and their column descriptions) |
| `*_physio.*`, `*_physioevents.*` | notebook 02 (the conversion) and notebook 05 (the sync, and the copy into `meg/`) |

## 5. Next steps

* **No errors?** The dataset is valid BIDS. Warnings are worth a read, but none of them make it invalid.
* **Sharing the dataset outside the lab** (e.g. on OpenNeuro): use the defaced MRI copies from notebook 03
  in place of the `anat/` images, and keep `DAYSBACK` (notebook 04) private.